# Entrega 2 — Entrenamiento y comparación de modelos
**Proyecto:** Telco Customer Churn · Despliegue de Soluciones Analíticas (MIAD)

**Objetivo.** Entrenar y comparar cuatro modelos supervisados para predecir `Churn` (0 = permanece; 1 = abandona), priorizando la detección de clientes en riesgo. Se conserva el procesamiento de datos del repositorio del equipo.

**Reproducción:** ejecutar las celdas en orden en Google Colab. El archivo CSV original debe cargarse manualmente; no se incluyen credenciales AWS ni datos de acceso. Los resultados visibles corresponden a la ejecución previa en Colab y deben verificarse al volver a ejecutar.

In [2]:
!git clone -b feature/karen-entrega2 https://github.com/t3indexpg/telco-churn.git /content/telco-churn

!git -C /content/telco-churn branch --show-current

Cloning into '/content/telco-churn'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 94 (delta 24), reused 73 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 6.66 MiB | 16.72 MiB/s, done.
Resolving deltas: 100% (24/24), done.
feature/karen-entrega2


In [3]:
from pathlib import Path

raiz = Path("/content/telco-churn")

archivos = {
    "Script de limpieza": raiz / "src/data/make_dataset.py",
    "Script de división": raiz / "src/data/split_data.py",
    "Configuración": raiz / "params.yaml",
    "Dataset original": raiz / "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv",
    "Datos de entrenamiento": raiz / "data/processed/train.csv",
    "Datos de prueba": raiz / "data/processed/test.csv"
}

for nombre, ruta in archivos.items():
    estado = "DISPONIBLE" if ruta.exists() else "FALTA"
    print(f"{nombre}: {estado}")

Script de limpieza: DISPONIBLE
Script de división: DISPONIBLE
Configuración: DISPONIBLE
Dataset original: FALTA
Datos de entrenamiento: FALTA
Datos de prueba: FALTA


In [5]:
from google.colab import files
from pathlib import Path
import shutil

archivo_subido = files.upload()

destino = Path(
    "/content/telco-churn/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

destino.parent.mkdir(parents=True, exist_ok=True)

nombre = next(iter(archivo_subido))
shutil.copy2(nombre, destino)

print("Dataset cargado correctamente:", destino.exists())

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn (1).csv
Dataset cargado correctamente: True


In [6]:
%cd /content/telco-churn

!python src/data/make_dataset.py
!python src/data/split_data.py

/content/telco-churn
Crudo: (7043, 21) -> Limpio: (7043, 27) | nulos: 0
Guardado en data/processed/telco_churn_limpio.csv
train: (5634, 27), churn=0.265
test:  (1409, 27), churn=0.265


## 1. Entorno y repositorio

In [11]:
%pip install -q imbalanced-learn joblib

from pathlib import Path
import pandas as pd
import shutil

ROOT = Path('/content/telco-churn')
if not (ROOT / 'src/data/make_dataset.py').exists():
    !git clone https://github.com/t3indexpg/telco-churn.git /content/telco-churn
print('Repositorio disponible:', ROOT.exists())

Repositorio disponible: True


## 2. Dataset original
Subir a Colab el archivo **`WA_Fn-UseC_-Telco-Customer-Churn.csv`** desde la sección **Archivos** (o usar el selector que aparece al ejecutar esta celda). El repositorio contiene un puntero DVC al archivo; esta carga manual evita depender de permisos temporales del bucket S3. El dataset debe tener **7.043 filas y 21 columnas**. Verificar que la versión del CSV corresponda a la usada por el equipo.

In [12]:
from google.colab import files
nombre_csv = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
destino = ROOT / 'data/raw' / nombre_csv
origen = Path('/content') / nombre_csv
if not origen.exists() and not destino.exists():
    print('Selecciona el CSV original de Telco Customer Churn:')
    files.upload()
if origen.exists():
    destino.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(origen, destino)
assert destino.exists(), f'No se encontró {nombre_csv}'
datos = pd.read_csv(destino)
print('Dimensiones del dataset:', datos.shape)
assert datos.shape == (7043, 21), 'Revisar la versión del archivo original'

Dimensiones del dataset: (7043, 21)


## 3. Limpieza y partición estratificada
Se utilizan los scripts del repositorio (`make_dataset.py` y `split_data.py`) con los parámetros del equipo: prueba 20 % y semilla 42.

In [8]:
%cd /content/telco-churn

!python src/data/make_dataset.py

/content/telco-churn
Crudo: (7043, 21) -> Limpio: (7043, 27) | nulos: 0
Guardado en data/processed/telco_churn_limpio.csv


In [9]:
%cd /content/telco-churn

!python src/data/split_data.py

/content/telco-churn
train: (5634, 27), churn=0.265
test:  (1409, 27), churn=0.265


In [7]:
import pandas as pd

train = pd.read_csv("/content/telco-churn/data/processed/train.csv")
test = pd.read_csv("/content/telco-churn/data/processed/test.csv")

print("Entrenamiento:", train.shape)
print("Prueba:", test.shape)
print("Variable objetivo:", train["Churn"].unique())
print("Clientes repetidos entre conjuntos:",
      len(set(train["customerID"]) & set(test["customerID"])))

Entrenamiento: (5634, 27)
Prueba: (1409, 27)
Variable objetivo: [0 1]
Clientes repetidos entre conjuntos: 0


## 4. Preprocesamiento
Las transformaciones se ajustan exclusivamente con los datos de entrenamiento mediante un `Pipeline`. Se excluyen `customerID` y `Churn` de las variables predictoras.

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Separar variables predictoras y objetivo
X_train = train.drop(columns=["Churn", "customerID"])
y_train = train["Churn"]

X_test = test.drop(columns=["Churn", "customerID"])
y_test = test["Churn"]

# Identificar tipos de variables
columnas_numericas = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

columnas_categoricas = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Preparación de variables numéricas
transformador_numerico = Pipeline([
    ("imputador", SimpleImputer(strategy="median")),
    ("escalador", StandardScaler())
])

# Preparación de variables categóricas
transformador_categorico = Pipeline([
    ("imputador", SimpleImputer(strategy="most_frequent")),
    ("codificador", OneHotEncoder(handle_unknown="ignore"))
])

# Preprocesamiento general
preprocesador = ColumnTransformer([
    ("numericas", transformador_numerico, columnas_numericas),
    ("categoricas", transformador_categorico, columnas_categoricas)
])

print("Variables numéricas:", len(columnas_numericas))
print("Variables categóricas:", len(columnas_categoricas))
print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)
print("Preprocesador configurado correctamente.")

Variables numéricas: 6
Variables categóricas: 19
Entrenamiento: (5634, 25)
Prueba: (1409, 25)
Preprocesador configurado correctamente.


## 5. Modelos supervisados

### 5.1 Regresión logística estándar

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Crear el modelo
modelo_logistico = Pipeline([
    ("preprocesamiento", preprocesador),
    ("clasificador", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Entrenar únicamente con los datos de entrenamiento
modelo_logistico.fit(X_train, y_train)

# Generar predicciones sobre los datos de prueba
y_pred = modelo_logistico.predict(X_test)
y_prob = modelo_logistico.predict_proba(X_test)[:, 1]

# Calcular métricas
metricas_logistico = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-score": f1_score(y_test, y_pred),
    "AUC-ROC": roc_auc_score(y_test, y_prob)
}

print("RESULTADOS - REGRESIÓN LOGÍSTICA")
print("-" * 40)

for nombre, valor in metricas_logistico.items():
    print(f"{nombre}: {valor:.4f}")

print("\nMATRIZ DE CONFUSIÓN")
print(confusion_matrix(y_test, y_pred))

print("\nREPORTE DE CLASIFICACIÓN")
print(classification_report(y_test, y_pred))

RESULTADOS - REGRESIÓN LOGÍSTICA
----------------------------------------
Accuracy: 0.7984
Precision: 0.6562
Recall: 0.5053
F1-score: 0.5710
AUC-ROC: 0.8448

MATRIZ DE CONFUSIÓN
[[936  99]
 [185 189]]

REPORTE DE CLASIFICACIÓN
              precision    recall  f1-score   support

           0       0.83      0.90      0.87      1035
           1       0.66      0.51      0.57       374

    accuracy                           0.80      1409
   macro avg       0.75      0.70      0.72      1409
weighted avg       0.79      0.80      0.79      1409



### 5.2 Regresión logística balanceada

In [15]:
modelo_balanceado = Pipeline([
    ("preprocesamiento", preprocesador),
    ("clasificador", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

modelo_balanceado.fit(X_train, y_train)

y_pred_bal = modelo_balanceado.predict(X_test)
y_prob_bal = modelo_balanceado.predict_proba(X_test)[:, 1]

metricas_balanceado = {
    "Accuracy": accuracy_score(y_test, y_pred_bal),
    "Precision": precision_score(y_test, y_pred_bal),
    "Recall": recall_score(y_test, y_pred_bal),
    "F1-score": f1_score(y_test, y_pred_bal),
    "AUC-ROC": roc_auc_score(y_test, y_prob_bal)
}

print("RESULTADOS - REGRESIÓN LOGÍSTICA BALANCEADA")
print("-" * 45)

for nombre, valor in metricas_balanceado.items():
    print(f"{nombre}: {valor:.4f}")

print("\nMATRIZ DE CONFUSIÓN")
print(confusion_matrix(y_test, y_pred_bal))

RESULTADOS - REGRESIÓN LOGÍSTICA BALANCEADA
---------------------------------------------
Accuracy: 0.7388
Precision: 0.5050
Recall: 0.8048
F1-score: 0.6206
AUC-ROC: 0.8450

MATRIZ DE CONFUSIÓN
[[740 295]
 [ 73 301]]


### 5.3 Regresión logística con SMOTE
SMOTE se aplica solo durante el entrenamiento, después de la codificación. **Limitación:** al interpolar variables codificadas con one-hot puede producir combinaciones sintéticas no equivalentes a categorías originales; evaluar SMOTENC en futuras iteraciones.

In [16]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

modelo_smote = ImbPipeline([
    ("preprocesamiento", preprocesador),
    ("smote", SMOTE(random_state=42)),
    ("clasificador", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

modelo_smote.fit(X_train, y_train)

y_pred_smote = modelo_smote.predict(X_test)
y_prob_smote = modelo_smote.predict_proba(X_test)[:, 1]

metricas_smote = {
    "Accuracy": accuracy_score(y_test, y_pred_smote),
    "Precision": precision_score(y_test, y_pred_smote),
    "Recall": recall_score(y_test, y_pred_smote),
    "F1-score": f1_score(y_test, y_pred_smote),
    "AUC-ROC": roc_auc_score(y_test, y_prob_smote)
}

print("RESULTADOS - REGRESIÓN LOGÍSTICA CON SMOTE")
print("-" * 45)

for nombre, valor in metricas_smote.items():
    print(f"{nombre}: {valor:.4f}")

print("\nMATRIZ DE CONFUSIÓN")
print(confusion_matrix(y_test, y_pred_smote))

RESULTADOS - REGRESIÓN LOGÍSTICA CON SMOTE
---------------------------------------------
Accuracy: 0.7438
Precision: 0.5111
Recall: 0.7995
F1-score: 0.6236
AUC-ROC: 0.8433

MATRIZ DE CONFUSIÓN
[[749 286]
 [ 75 299]]


### 5.4 Random Forest con SMOTE

In [17]:
from sklearn.ensemble import RandomForestClassifier

modelo_rf_smote = ImbPipeline([
    ("preprocesamiento", preprocesador),
    ("smote", SMOTE(random_state=42)),
    ("clasificador", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

modelo_rf_smote.fit(X_train, y_train)

y_pred_rf = modelo_rf_smote.predict(X_test)
y_prob_rf = modelo_rf_smote.predict_proba(X_test)[:, 1]

metricas_rf = {
    "Accuracy": accuracy_score(y_test, y_pred_rf),
    "Precision": precision_score(y_test, y_pred_rf),
    "Recall": recall_score(y_test, y_pred_rf),
    "F1-score": f1_score(y_test, y_pred_rf),
    "AUC-ROC": roc_auc_score(y_test, y_prob_rf)
}

print("RESULTADOS - RANDOM FOREST CON SMOTE")
print("-" * 45)

for nombre, valor in metricas_rf.items():
    print(f"{nombre}: {valor:.4f}")

print("\nMATRIZ DE CONFUSIÓN")
print(confusion_matrix(y_test, y_pred_rf))

RESULTADOS - RANDOM FOREST CON SMOTE
---------------------------------------------
Accuracy: 0.7757
Precision: 0.5801
Recall: 0.5615
F1-score: 0.5707
AUC-ROC: 0.8248

MATRIZ DE CONFUSIÓN
[[883 152]
 [164 210]]


## 6. Comparación final

In [18]:
comparacion_final = pd.DataFrame([
    {"Modelo": "Regresión logística", **metricas_logistico},
    {"Modelo": "Regresión logística balanceada", **metricas_balanceado},
    {"Modelo": "Regresión logística SMOTE", **metricas_smote},
    {"Modelo": "Random Forest SMOTE", **metricas_rf}
])

comparacion_final = comparacion_final.sort_values(
    by="Recall",
    ascending=False
).reset_index(drop=True)

print("COMPARACIÓN FINAL - ENTREGA 2")
display(comparacion_final.round(4))

# Guardar resultados para el informe
ruta_resultados = "/content/telco-churn/reports/comparacion_modelos_entrega2.csv"

comparacion_final.to_csv(ruta_resultados, index=False)

print("\nResultados guardados en:", ruta_resultados)

COMPARACIÓN FINAL - ENTREGA 2


,Modelo,Accuracy,Precision,Recall,F1-score,AUC-ROC
0,Regresión logística balanceada,0.7388,0.5050,0.8048,0.6206,0.8450
1,Regresión logística SMOTE,0.7438,0.5111,0.7995,0.6236,0.8433
2,Random Forest SMOTE,0.7757,0.5801,0.5615,0.5707,0.8248
3,Regresión logística,0.7984,0.6562,0.5053,0.5710,0.8448



Resultados guardados en: /content/telco-churn/reports/comparacion_modelos_entrega2.csv


### Interpretación
En la ejecución documentada, la regresión logística balanceada obtuvo el mayor **recall (0,8048)** y el mayor **AUC-ROC (0,8450)**; la regresión logística con SMOTE alcanzó el mayor **F1-score (0,6236)**. La elección de modelo debe considerar el costo de falsas alarmas y clientes no detectados.

**Trazabilidad:** los cuatro modelos se entrenaron en Colab y posteriormente se registraron en el experimento `Telco_Churn_Entrega2` de MLflow alojado en AWS EC2. El registro remoto se realizó por separado, por lo que este notebook no necesita credenciales ni conexión directa a MLflow.

## 7. Guardado de modelos

In [19]:
import joblib
from pathlib import Path

carpeta_modelos = Path("/content/modelos_entrega2")
carpeta_modelos.mkdir(exist_ok=True)

modelos = {
    "regresion_logistica": modelo_logistico,
    "regresion_balanceada": modelo_balanceado,
    "regresion_smote": modelo_smote,
    "random_forest_smote": modelo_rf_smote
}

for nombre, modelo in modelos.items():
    ruta = carpeta_modelos / f"{nombre}.joblib"
    joblib.dump(modelo, ruta)
    print(f"Guardado: {ruta.name}")

print("\nTotal de modelos guardados:", len(list(carpeta_modelos.glob("*.joblib"))))

Guardado: regresion_logistica.joblib
Guardado: regresion_balanceada.joblib
Guardado: regresion_smote.joblib
Guardado: random_forest_smote.joblib

Total de modelos guardados: 4


**Opcional:** descargar los modelos para su almacenamiento o traslado seguro. Los archivos `.joblib` solo deben cargarse desde fuentes confiables. Los modelos no se suben al repositorio Git por su tamaño; MLflow almacena los artefactos.

In [ ]:
# Descarga opcional desde Colab (no se ejecuta automáticamente)
# from google.colab import files
# ruta_zip = shutil.make_archive('/content/modelos_entrega2', 'zip', '/content/modelos_entrega2')
# files.download(ruta_zip)